In [8]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.metrics import r2_score
from sklearn.ensemble import StackingRegressor,GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.compose import make_column_selector,make_column_transformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from tqdm import tqdm
import os
os.chdir('E:/Machine_Learning/Cases/Medical_Cost_Personal/')

In [2]:
df = pd.read_csv('insurance.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
X,y = df.drop('charges',axis = 1),df['charges']
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.3,random_state=26)

ohe = OneHotEncoder(sparse_output= False,drop='first')
ct = make_column_transformer((ohe,make_column_selector(dtype_include='object')),
                            ('passthrough',make_column_selector(dtype_include=['int64','float64'])),
                            verbose_feature_names_out=False)
ct = ct.set_output(transform='pandas')
X_train_ohe = ct.fit_transform(X_train)
X_test_ohe = ct.transform(X_test)

In [6]:
lr = LinearRegression()
ridge = Ridge()
lasso = Lasso()
dtr  = DecisionTreeRegressor(random_state=26)
knn = KNeighborsRegressor()
scaler = StandardScaler()
pipe_knn = Pipeline([('SCL',scaler),('KNN',knn)])
gbr = GradientBoostingRegressor(random_state=26)

In [9]:
stack = StackingRegressor([('LR',lr),('Ridge',ridge),('Lasso',lasso),('TREE',dtr),('KNN',pipe_knn)],final_estimator=gbr)
stack.fit(X_train_ohe,y_train)
y_pred = stack.predict(X_test_ohe)
r2_score(y_test,y_pred)

0.8535975448647124

### Another way

In [ ]:
X,y = df.drop('charges',axis = 1),df['charges']
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.3,random_state=26)

ohe = OneHotEncoder(sparse_output= False,drop='first')
ct = make_column_transformer((ohe,make_column_selector(dtype_include='object')),
                            ('passthrough',make_column_selector(dtype_include=['int64','float64'])),
                            verbose_feature_names_out=False)
ct = ct.set_output(transform='pandas')

In [11]:
lr = LinearRegression()
ridge = Ridge()
lasso = Lasso()
dtr  = DecisionTreeRegressor(random_state=26)
knn = KNeighborsRegressor()
scaler = StandardScaler()
pipe_knn = Pipeline([('SCL',scaler),('KNN',knn)])
gbr = GradientBoostingRegressor(random_state=26)

stack = StackingRegressor([('LR',lr),('Ridge',ridge),('Lasso',lasso),('TREE',dtr),('KNN',pipe_knn)],final_estimator=gbr)

pipe_whole = Pipeline([('OHE',ct),('Stack',stack)]) # second pipeline for OHE and stack (so no need to create multiple pipeline for each model for OHE)
pipe_whole.fit(X_train,y_train)
y_pred = pipe_whole.predict(X_test)
r2_score(y_test,y_pred)

0.8535975448647124

## XGBoost Regressor

In [12]:
from xgboost import XGBRegressor

In [14]:
xgb = XGBRegressor(random_state=26)

In [15]:
stack = StackingRegressor([('LR',lr),('Ridge',ridge),('Lasso',lasso),('TREE',dtr),('KNN',pipe_knn)],final_estimator=xgb)
stack.fit(X_train_ohe,y_train)
y_pred = stack.predict(X_test_ohe)
r2_score(y_test,y_pred)

0.8112013667665723